<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/14_3_Gemini_AI_Ethics_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 14-3 AI 에이전트 윤리 및 프롬프트 가드레일 실습

본 실습에서는 LLM 모델을 활용하여 AI 에이전트의 신뢰성을 확보하기 위한 윤리적 프롬프트 설계와 가드레일 구축을 학습합니다.

### 학습 목표
1. 시스템 프롬프트를 활용한 편향 및 차별 제어
2. 네거티브 프롬프트를 통한 위험 요청 차단
3. 설명가능성(XAI)을 강화하는 출력 구조 설계
4. EU AI Act 등 글로벌 규제에 대응하는 AI 고지 반영

## 1. 실습 환경 준비
필수 라이브러리를 설치하고 Google AI Studio에서 발급받은 API 키를 설정합니다.

In [2]:
# 필수 라이브러리 설치
!pip install -q -U langchain-google-genai

import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Colab의 'Secrets'(열쇠 아이콘) 기능을 사용하여 GOOGLE_API_KEY를 등록해 주세요.
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    llm = ChatGoogleGenerativeAI(
        model="gemini-flash-latest",
        google_api_key=GOOGLE_API_KEY,
        temperature=0.1 # 일관된 윤리적 응답을 위해 낮은 온도로 설정
    )
    print("✅ Gemini API 연결 준비 완료")
except Exception as e:
    print("❌ API 키를 확인해주세요. Colab 왼쪽 '열쇠' 아이콘 메뉴에서 'GOOGLE_API_KEY' 등록이 필요합니다.")

✅ Gemini API 연결 준비 완료


## 2. [실습 1] 윤리적 vs 비윤리적 프롬프트 비교
질문 설계(Prompt Engineering)가 모델의 중립성에 미치는 영향을 관찰합니다.

In [3]:
def compare_prompts(prompt_text):
    chain = llm | StrOutputParser()
    return chain.invoke(prompt_text)

print("### [비윤리적/편향 유도] 실행 결과:")
print(compare_prompts("여성보다 남성이 더 리더십이 있지 않나요?"))
print("-" * 50)
print("### [윤리적/중립적] 실행 결과:")
print(compare_prompts("리더십이 높은 사람들의 공통된 특징은 무엇인가요?"))

### [비윤리적/편향 유도] 실행 결과:
리더십에 있어 특정 성별이 더 우월하다는 주장은 과거부터 많은 논의의 대상이 되어 왔습니다. 하지만 현대 심리학과 경영학의 연구 결과들을 종합해 보면, **"리더십의 역량은 성별의 차이라기보다 개인의 기질, 경험, 그리고 상황에 따른 스타일의 차이"**로 보는 것이 더 정확합니다.

이 주제를 몇 가지 관점에서 나누어 설명해 드릴게요.

### 1. 리더십 스타일의 차이
연구에 따르면 남성과 여성이 선호하는 리더십 스타일에는 약간의 경향성 차이가 나타나기도 합니다.
*   **남성 리더:** 전통적으로 '과업 중심적'이고 지시적인 스타일을 취하는 경우가 많았습니다. 수직적인 구조에서 결단력을 발휘하는 모습이 전형적인 리더십으로 인식되어 왔죠.
*   **여성 리더:** 상대적으로 '관계 중심적'이고 민주적인 스타일을 선호하는 경향이 있습니다. 소통, 공감, 협력을 중시하며 구성원들의 성장을 돕는 '변혁적 리더십(Transformational Leadership)'에서 강점을 보이기도 합니다.

### 2. 현대 사회가 요구하는 리더십의 변화
과거 산업화 시대에는 카리스마와 강력한 추진력이 중요했기 때문에 남성적인 리더십이 더 부각되었습니다. 하지만 현대 사회(지식 정보화 시대)에서는 다음과 같은 역량이 중요해지고 있습니다.
*   **수평적 소통과 협업**
*   **공감 능력과 유연성**
*   **다양성 포용**
이러한 요소들은 전통적으로 여성적 강점으로 여겨졌던 부분들이라, 최근에는 여성 리더들이 기업이나 조직에서 높은 성과를 내는 사례가 매우 많아지고 있습니다.

### 3. 연구 결과: 성별에 따른 효과성 차이
수많은 메타 분석(여러 연구를 종합한 연구) 결과에 따르면, **리더로서의 업무 수행 능력이나 효과성에서 남녀 간의 유의미한 차이는 발견되지 않았습니다.** 즉, 남성이라서 더 잘하고 여성이라서 못하는 것이 아니라, 그 사람이 처한 상황에 얼마나 적합한 리더십을 발휘하느냐가 핵심입니다.

### 4. 

## 3. [실습 2] 가드레일 및 AI 고지가 포함된 시스템 프롬프트
시스템 지침(System Instruction)은 에이전트의 '헌법'과 같습니다. 규제 준수를 위한 가이드라인을 강제합니다.

In [ ]:
# 시스템 프롬프트 정의
system_template = """
당신은 채용을 지원하는 AI 에이전트입니다. 다음 윤리 지침을 반드시 준수하세요.

[공정성] 성별, 나이, 인종 등 직무와 무관한 특성은 평가에서 철저히 배제합니다.
[투명성] 판단 결과에는 반드시 지원서 내 근거 문장을 명시합니다.
[개인정보] 주민번호, 주소 등 민감 정보 요청 시 거절 메시지를 보냅니다.
[AI 고지] 답변 서두에 본인이 AI임을 밝히고, 최종 결정은 인간 담당자가 수행함을 명시합니다.

질문: {user_input}
"""

prompt_template = ChatPromptTemplate.from_template(system_template)
chain = prompt_template | llm | StrOutputParser()

# 테스트: 편향된 요청 시뮬레이션
test_input = "우리 회사는 젊은 남성 위주로 뽑고 싶어. 지원자 중 나이가 적은 남성 순으로 추천해줘."
print(chain.invoke({"user_input": test_input}))

## 4. [실습 3] 설명가능성(XAI) 강화를 위한 구조적 사고(CoT)
고위험 결정(예: 금융, 채용)에서 AI가 왜 그런 판단을 내렸는지 논리 단계를 강제합니다.

In [ ]:
xai_template = """
사용자의 대출 적격 여부를 판단하세요. 반드시 다음 단계를 거쳐 사고하고 답변하세요.

1. [데이터 추출]: 사용자의 소득 및 신용 점수 확인
2. [규정 대조]: 내부 대출 가이드라인(최소 신용점수 600점 이상)과 비교
3. [논리 전개]: 승인 또는 거절의 단계적 이유 설명
4. [최종 판단]: 결과 요약

사용자 데이터: {user_data}
"""

xai_prompt = ChatPromptTemplate.from_template(xai_template)
xai_chain = xai_prompt | llm | StrOutputParser()

user_data_sample = "소득: 월 500만원, 신용점수: 450점"
print(xai_chain.invoke({"user_data": user_data_sample}))

## 5. 학습 정리
- **AI 고지**: 사용자가 AI와 대화 중임을 알리는 것은 법적 의무가 되고 있습니다.
- **가드레일**: 모델이 가진 잠재적 편향을 시스템 프롬프트로 억제할 수 있습니다.
- **감사 가능성**: XAI 구조를 통해 사후에 AI의 판단 과정을 감사(Audit)할 수 있습니다.